In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import scipy.interpolate as interp
import cartopy.crs as ccrs
from cartopy.mpl.ticker import (LongitudeFormatter, LatitudeFormatter,
                                LongitudeLocator, LatitudeLocator)
from matplotlib.colors import Normalize
import matplotlib.patches as mpatches
import openpyxl

alpha = '\u03B1'
beta = '\u03B2'
titles = {
    'apiose' : f'1-MeO-{beta}-D-Api',
    'arabinofur_a' : f'1-MeO-{alpha}-L-Araf',
    'arabinofur_b' : f'1-MeO-{beta}-L-Araf',
    'aceric' : f'1,2-MeO-{alpha}-L-AceA',
    'arabinopyr' : f'1-MeO-{alpha}-L-Arap',
    'arabinopyr123' : f'1,2,3-MeO-{alpha}-L-Arap',
    'Dgalactose' : f'1,2,4-MeO-{beta}-D-Gal',
    'dha' : f'1,4-MeO-Dha',
    'fucose12' : f'1,2-MeO-{alpha}-L-Fuc',
    'fucose134' : f'1,3,4-MeO-{alpha}-L-Fuc',
    'galacturonic_b' : f'1-MeO-{beta}-D-GalA',
    'galacturonic' : f'{alpha}-D-GalA',
    'galacturonic4_a' : f'4-MeO-{alpha}-D-GalA',
    'galacturonic1_a' : f'1-MeO-{alpha}-D-GalA',
    'galacturonic14' : f'1,4-MeO-{alpha}-D-GalA',
    'galacturonic124' : f'1,2,4-MeO-{alpha}-D-GalA',
    'galacturonic134' : f'1,3,4-MeO-{alpha}-D-GalA',
    'galacturonic1234' : f'1,2,3,4-MeO-{alpha}-D-GalA',
    'glucuronic' : f'1,2-MeO-{beta}-D-GlcA',
    'kdo' : f'1,4-MeO-Kdo',
    'Lgalactose' : f'1-MeO-{alpha}-L-Gal',
    'rhamnose1' : f'1-MeO-{alpha}-L-Rha',
    'rhamnose13' : f'1,3-MeO-{alpha}-L-Rha',
    'rhamnose1234' : f'1,2,3,4-MeO-{alpha}-L-Rha',
    'xylose' : f'1,2-MeO-{alpha}-D-Xyl'
}

pucker_info_5_mem_dict = {
    '1e' : [r'$^{1}E$', 0, 0, 0, 0, 0, 0],
    '1t2' : [r'${}^1T_2$', 0, 0, 0, 0, 0, 0],
    'e2' : [r'$E_{2}$', 0, 0, 0, 0, 0, 0],
    '3t2' : [r'${}^3T_2$', 0, 0, 0, 0, 0, 0],
    '3e' : [r'$^{3}E$', 0, 0, 0, 0, 0, 0],
    '3t4' : [r'${}^3T_4$', 0, 0, 0, 0, 0, 0],
    'e4' : [r'$E_{4}$', 0, 0, 0, 0, 0, 0],
    '5t4' : [r'${}^0T_4$', 0, 0, 0, 0, 0, 0],
    '5e' : [r'$^{0}E$', 0, 0, 0, 0, 0, 0],
    '5t1' : [r'${}^0T_1$', 0, 0, 0, 0, 0, 0],
    'e1' : [r'$E_{1}$', 0, 0, 0, 0, 0, 0],
    '2t1' : [r'${}^2T_1$', 0, 0, 0, 0, 0, 0],
    '2e' : [r'$^{2}E$', 0, 0, 0, 0, 0, 0],
    '2t3' : [r'${}^2T_3$', 0, 0, 0, 0, 0, 0],
    'e3' : [r'$E_{3}$', 0, 0, 0, 0, 0, 0],
    '4t3' : [r'${}^4T_3$', 0, 0, 0, 0, 0, 0],
    '4e' : [r'$^{4}E$', 0, 0, 0, 0, 0, 0],
    '4t5' : [r'${}^4T_0$', 0, 0, 0, 0, 0, 0],
    'e5' : [r'$E_{0}$', 0, 0, 0, 0, 0, 0],
    '1t5' : [r'${}^1T_0$', 0, 0, 0, 0, 0, 0]
}

pucker_info_6_mem_dict = {
    '1C4' : [r'${}^1C_4$', 0, 0, 0, 0, 0, 0],
    '3E' : [r'$^{3}E$', 0, 0, 0, 0, 0, 0],
    '3H4' : [r'${}^3H_4$', 0, 0, 0, 0, 0, 0],
    'E4' : [r'$E_{4}$', 0, 0, 0, 0, 0, 0],
    '5H4' : [r'${}^5H_4$', 0, 0, 0, 0, 0, 0],
    '5E' : [r'$^{5}E$', 0, 0, 0, 0, 0, 0],
    '5H6' : [r'${}^5H_0$', 0, 0, 0, 0, 0, 0],
    'E6' : [r'$E_{0}$', 0, 0, 0, 0, 0, 0],
    '1H6' : [r'${}^1H_0$', 0, 0, 0, 0, 0, 0],
    '1E' : [r'$^{1}E$', 0, 0, 0, 0, 0, 0],
    '1H2' : [r'${}^1H_2$', 0, 0, 0, 0, 0, 0],
    'E2' : [r'$E_{2}$', 0, 0, 0, 0, 0, 0],
    '3H2' : [r'${}^3H_2$', 0, 0, 0, 0, 0, 0],
    '36B' : [r'$^{0,3}B$', 0, 0, 0, 0, 0, 0],
    '3S1' : [r'${}^3S_1$', 0, 0, 0, 0, 0, 0],
    'B14' : [r'$B_{1,4}$', 0, 0, 0, 0, 0, 0],
    '5S1' : [r'${}^5S_1$', 0, 0, 0, 0, 0, 0],
    '25B' : [r'$^{2,5}B$', 0, 0, 0, 0, 0, 0],
    '2S6' : [r'${}^2S_0$', 0, 0, 0, 0, 0, 0],
    'B36' : [r'$B_{0,3}$', 0, 0, 0, 0, 0, 0],
    '1S3' : [r'${}^1S_3$', 0, 0, 0, 0, 0, 0],
    '14B' : [r'$^{1,4}B$', 0, 0, 0, 0, 0, 0],
    '1S5' : [r'${}^1S_5$', 0, 0, 0, 0, 0, 0],
    'B25' : [r'$B_{2,5}$', 0, 0, 0, 0, 0, 0],
    '6S2' : [r'${}^0S_2$', 0, 0, 0, 0, 0, 0],
    '6E' : [r'$^{0}E$', 0, 0, 0, 0, 0, 0],
    '6H1' : [r'${}^0H_1$', 0, 0, 0, 0, 0, 0],
    'E1' : [r'$E_{1}$', 0, 0, 0, 0, 0, 0],
    '2H1' : [r'${}^2H_1$', 0, 0, 0, 0, 0, 0],
    '2E' : [r'$^{2}E$', 0, 0, 0, 0, 0, 0],
    '2H3' : [r'${}^2H_3$', 0, 0, 0, 0, 0, 0],
    'E3' : [r'$E_{3}$', 0, 0, 0, 0, 0, 0],
    '4H3' : [r'${}^4H_3$', 0, 0, 0, 0, 0, 0],
    '4E' : [r'$^{4}E$', 0, 0, 0, 0, 0, 0],
    '4H5' : [r'${}^4H_5$', 0, 0, 0, 0, 0, 0],
    'E5' : [r'$E_{5}$', 0, 0, 0, 0, 0, 0],
    '6H5' : [r'${}^0H_5$', 0, 0, 0, 0, 0, 0],
    '4C1' : [r'${}^4C_1$', 0, 0, 0, 0, 0, 0]
}

pucker_info_5_mem = pd.DataFrame.from_dict(pucker_info_5_mem_dict, orient='index')
column_names = ['Pucker State', 'Lowest Energy #1', 'Lowest Energy #2', 'Lowest Energy #3',
                               'Highest Energy #1', 'Highest Energy #2', 'Highest Energy #3']
pucker_info_5_mem.columns = column_names

pucker_info_6_mem = pd.DataFrame.from_dict(pucker_info_6_mem_dict, orient='index')
pucker_info_6_mem.columns = column_names

In [ ]:
# Save figure in specified folder
def save_fig(save, output_path, sugar, output_subfolder):
    graph_output_path = os.path.join(output_path, output_subfolder)
    os.system(f'mkdir {graph_output_path}')
    if save == 'png' or save == 'both':
        output_graph_png = os.path.join(graph_output_path, f'{sugar}_scatter.png')
        plt.savefig(output_graph_png, format='png', dpi=600, transparent=True)
    if save == 'svg' or save == 'both':
        output_graph_svg = os.path.join(graph_output_path, f'{sugar}_scatter.svg')
        plt.savefig(output_graph_svg, format='svg', transparent=True)

# Save pdbs for three highest and three lowest energy structures
def sort_energy(energy):
    sorted_intensity = np.argsort(energy)
    return sorted_intensity

# Converts DataFrame into numpy array for 6-mem ring; prerequisite for graphing
def return_numpy_6_mem(data, energy_type):
    energy = data[energy_type].to_numpy()
    phi_coords = data['Phi'].to_numpy()
    phi_coords = np.copy(phi_coords)
    phi_coords -= 165 # Shift each point by 15 degrees to the right so that it is visible on the Mollweide plot

    # If any points are greater than 180 (outside the plot area), then force them to wrap around
    for i, val in enumerate(phi_coords):
        if val > 180:
            phi_coords[i] -= 360
    theta_coords = data['Theta'].to_numpy()
    theta_coords = np.copy(theta_coords)
    theta_coords -= 90
    pucker_states = data['Pucker State'].to_numpy()

    return energy, phi_coords, theta_coords, pucker_states

# Converts DataFrame into numpy array for 5-mem ring; prerequisite for graphing
def return_numpy_5_mem(data, energy_type):
    energy = data[energy_type].to_numpy()
    phi_coords = data['Phi'].to_numpy()
    phi_coords = np.copy(phi_coords)
    phi_coords -= 144
    phi_radians = np.deg2rad(phi_coords)
    Q = data['Q'].to_numpy()
    pucker_states = data['Pucker State'].to_numpy()

    return energy, phi_radians, Q, pucker_states

# Gets correct x-axis label based on desired energy type
def get_symbol(energy_type):
    if energy_type == 'E Plus Thermal Enthalpy Norm.':
        symbol = 'H'
    elif energy_type == 'E Plus Free Energy Norm.':
        symbol = 'G'
    elif energy_type == 'E Plus ZPE Norm.' or energy_type == 'E Norm.':
        symbol = 'E'
    else:
        symbol = ''
    
    return symbol

# Generates interpolated background plot that is overlaid with discrete energy markers
def generate_base_plot(phi_coords, theta_coords, energy, energy_type):
    # Create Mollweide plot
    mollweide_proj = ccrs.Mollweide()
    mollweide_proj._threshold /= 100.0
    fig, ax = plt.subplots(figsize=(10,8), subplot_kw={'projection': mollweide_proj})
    gl = ax.gridlines(draw_labels=False, x_inline=True, xpadding=0, ypadding=0, zorder=1, color='gray', alpha=0.5)
    gl.xlocator = plt.FixedLocator([-165, -135, -105, -75, -45, -15, 15, 45, 75, 105, 135, 165])
    [-180, -150, -120, -90, -60, -30, 0, 30, 60, 90, 120, 150, 180]
    gl.ylocator = plt.FixedLocator(np.arange(-90, 91, 30))

    # Annotate longitude labels
    plt.annotate('0°', (-165, -56), ha='center', va='center', fontsize=16, transform=ccrs.PlateCarree())
    plt.annotate('60°', (-105, -56), ha='center', va='center', fontsize=16, transform=ccrs.PlateCarree())
    plt.annotate('120°', (-45, -56), ha='center', va='center', fontsize=16, transform=ccrs.PlateCarree())
    plt.annotate('180°', (15, -56), ha='center', va='center', fontsize=16, transform=ccrs.PlateCarree())
    plt.annotate('240°', (75, -56), ha='center', va='center', fontsize=16, transform=ccrs.PlateCarree())
    plt.annotate('300°', (135, -56), ha='center', va='center', fontsize=16, transform=ccrs.PlateCarree())
    plt.annotate('\nΦ', (0, -76), ha='center', va='top', fontsize=16, transform=ccrs.PlateCarree())

    # Annotate latitude labels
    plt.annotate('30°    ', (-180, -60), ha='right', va='center', fontsize=16, transform=ccrs.PlateCarree())
    plt.annotate('60°  ', (-180, -30), ha='right', va='center', fontsize=16, transform=ccrs.PlateCarree())
    plt.annotate('90° ', (-180, 0), ha='right', va='center', fontsize=16, transform=ccrs.PlateCarree())
    plt.annotate('120° ', (-180, 30), ha='right', va='center', fontsize=16, transform=ccrs.PlateCarree())
    plt.annotate('150°  ', (-180, 60), ha='right', va='center', fontsize=16, transform=ccrs.PlateCarree())
    plt.annotate(' θ', (180, 0), ha='left', va='center', fontsize=16, transform=ccrs.PlateCarree())

    # Set default font to Helvetica
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Helvetica']
    plt.rcParams.update({'mathtext.default': 'regular'})

    points = np.column_stack((phi_coords.ravel(), theta_coords.ravel()))

    # Meshgrid (1000 by 1000 points) that controls the resolution of interpolation
    # Smaller mesh values result in a coarser (less fine) interpolation
    grid_x, grid_y = np.meshgrid(
        np.linspace(-180, 180, 1000),
        np.linspace(-90, 90, 1000)
    )
    grid_points = np.column_stack((grid_x.ravel(), grid_y.ravel()))

    # INTERPOLATION using radial bias function
    rbf_output = interp.RBFInterpolator(points, energy, kernel='thin_plate_spline')
    z_rbf = rbf_output(grid_points)
    z_rbf = z_rbf.reshape(grid_x.shape)

    vmin = 0
    vmax = 25
    norm_energy = Normalize(vmin=vmin, vmax=vmax)

    ax.imshow(
        z_rbf,
        extent=(-180, 180, -90, 90),
        origin='lower',
        cmap='RdYlBu_r',
        norm=norm_energy,
        interpolation='none',
        alpha=0.25,
        transform=ccrs.PlateCarree()
    )

    ticks = []
    for i in range(vmin, vmax+5, 5):
        ticks.append(i)
        
    scatter = ax.scatter([], [], c=[], cmap='RdYlBu_r', transform=ccrs.PlateCarree())
    cbar = fig.colorbar(scatter, ax=ax, ticks=ticks, orientation='horizontal', pad=0.06)
    cbar.ax.tick_params(labelsize=30)
    symbol = get_symbol(energy_type)
    cbar.set_label(f'\u0394{symbol} (kcal/mol)', fontsize=30, labelpad=5)
    scatter.set_clim(vmin, vmax)
    plt.tight_layout()

    # Plot two dummy points to ensure that the plot spans from -180 to 180 in phi and -90 to 90 in theta
    ax.scatter(-180, 0, transform=ccrs.PlateCarree(), alpha=0)
    ax.scatter(180, 0, transform=ccrs.PlateCarree(), alpha=0)

    return fig, ax, norm_energy

def plot_6_mem_scatter(data, unconstrained_data, energy_type, title):
    # Extract columns from the dataframe and store as numpy array for easier handling
    energy, phi_coords, theta_coords, pucker_states = return_numpy_6_mem(data, energy_type)

    if not unconstrained_data.empty:
        energy_unconstrained, phi_coords_unconstrained, theta_coords_unconstrained, pucker_states_unconstrained = return_numpy_6_mem(unconstrained_data, energy_type)
        for i, theta in enumerate(theta_coords_unconstrained):
            if 90 - abs(theta) < 16:
                phi_coords_unconstrained[i] = 15

    # Extract the three highest and lowest energy states so they can be plotted with special markers
    sorted_intensity_index = sort_energy(energy)
    fig, ax, norm_energy = generate_base_plot(phi_coords, theta_coords, energy, energy_type)

    for i in range(len(energy)):
        pucker = pucker_states[i]
        label = pucker_info_6_mem.loc[pucker, 'Pucker State']
        if pucker == '1C4' or pucker == '3E':
            plt.annotate(label, (phi_coords[i], theta_coords[i]), textcoords="offset points",
                        xytext=(0, -22), ha='center', va='center', fontsize=21, transform=ccrs.PlateCarree())
        elif pucker == '3H2':
            plt.annotate(label, (phi_coords[i], theta_coords[i]), textcoords="offset points",
                        xytext=(12, -23), ha='center', va='center', fontsize=21, transform=ccrs.PlateCarree())
        elif pucker == 'E2':
            plt.annotate(label, (phi_coords[i], theta_coords[i]), textcoords="offset points",
                        xytext=(-5, 21), ha='center', va='center', fontsize=21, transform=ccrs.PlateCarree())
        elif pucker == '3H4':
            plt.annotate(label, (phi_coords[i], theta_coords[i]), textcoords="offset points",
                        xytext=(6, 20), ha='center', va='center', fontsize=21, transform=ccrs.PlateCarree())
        elif pucker == 'E4':
            plt.annotate(label, (phi_coords[i], theta_coords[i]), textcoords="offset points",
                        xytext=(6, 21), ha='center', va='center', fontsize=21, transform=ccrs.PlateCarree())
        else:
            plt.annotate(label, (phi_coords[i], theta_coords[i]), textcoords="offset points",
                        xytext=(0, 21), ha='center', va='center', fontsize=21, transform=ccrs.PlateCarree())
                
        if unconstrained_data.empty:
            if i == sorted_intensity_index[37] or i == sorted_intensity_index[0]:
                ax.scatter(phi_coords[i], theta_coords[i], c=energy[i], cmap='RdYlBu_r', norm=norm_energy,
                        s=350, marker='s', edgecolors='black', transform=ccrs.PlateCarree(), zorder=3)
            elif i == sorted_intensity_index[36] or i == sorted_intensity_index[1]:
                ax.scatter(phi_coords[i], theta_coords[i], c=energy[i], cmap='RdYlBu_r', norm=norm_energy,
                        s=350, marker='X', edgecolors='black', transform=ccrs.PlateCarree(), zorder=3)
            elif i == sorted_intensity_index[35] or i == sorted_intensity_index[2]:
                ax.scatter(phi_coords[i], theta_coords[i], c=energy[i], cmap='RdYlBu_r', norm=norm_energy,
                        s=400, marker='*', edgecolors='black', transform=ccrs.PlateCarree(), zorder=3)
            else:
                ax.scatter(phi_coords[i], theta_coords[i], c=energy[i], cmap='RdYlBu_r', norm=norm_energy,
                        s=350, marker='o', edgecolors='black', transform=ccrs.PlateCarree(), zorder=3)
        else:
            ax.scatter(phi_coords[i], theta_coords[i], c='black', s=5, marker='o',
                       edgecolors='black', transform=ccrs.PlateCarree(), zorder=3)
            ax.scatter(phi_coords_unconstrained[i], theta_coords_unconstrained[i], c='green', norm=norm_energy,
                        s=100, marker='o', transform=ccrs.PlateCarree(), zorder=3)
            ax.plot((phi_coords[i], phi_coords_unconstrained[i]), (theta_coords[i], theta_coords_unconstrained[i]),
                    c='black', transform=ccrs.Geodetic(), lw=1.2)
            
    ax.set_title(title, fontsize=34, pad=20)
            
    return sorted_intensity_index, pucker_states, energy

def plot_5_mem_scatter(data, unconstrained_data, energy_type, title):
    energy, phi_radians, Q, pucker_states = return_numpy_5_mem(data, energy_type)

    if not unconstrained_data.empty:
        energy_unconstrained, phi_radians_unconstrained, Q_unconstrained, pucker_states_unconstrained = return_numpy_5_mem(unconstrained_data, energy_type)

    sorted_energy_index = sort_energy(energy)

    fig, ax = plt.subplots(1, 1, figsize=(10,10), subplot_kw={'projection': 'polar'})

    # Set default font to Helvetica
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Helvetica']
    plt.rcParams.update({'mathtext.default': 'regular'})

    vmin = 0
    vmax = 8
    norm_energy = Normalize(vmin=vmin, vmax=vmax)
    scatter = ax.scatter([], [], c=[], cmap='RdYlBu_r')
    for i in range(len(energy)):
        if i == sorted_energy_index[19] or i == sorted_energy_index[0]:
            ax.scatter(phi_radians[i], Q[i], c=energy[i], cmap='RdYlBu_r', norm=norm_energy,
                       s=450, marker='s', edgecolors='black', zorder=3)
        elif i == sorted_energy_index[18] or i == sorted_energy_index[1]:
            ax.scatter(phi_radians[i], Q[i], c=energy[i], cmap='RdYlBu_r', norm=norm_energy,
                       s=450, marker='X', edgecolors='black', zorder=3)
        elif i == sorted_energy_index[17] or i == sorted_energy_index[2]:
            ax.scatter(phi_radians[i], Q[i], c=energy[i], cmap='RdYlBu_r', norm=norm_energy,
                       s=500, marker='*', edgecolors='black', zorder=3)
        else:
            ax.scatter(phi_radians[i], Q[i], c=energy[i], cmap='RdYlBu_r', norm=norm_energy,
                       s=450, marker='o', edgecolors='black', zorder=3)
            
        if unconstrained_data.empty:
            pucker = pucker_states[i]
            label = pucker_info_5_mem.loc[pucker, 'Pucker State']
            plt.annotate(label, (phi_radians[i], Q[i]), textcoords="offset points",
                     xytext=(10, 19), ha='center', va='center', fontsize=20)
        else:
            ax.scatter(phi_radians_unconstrained[i], Q_unconstrained[i], c='green', norm=norm_energy,
                        s=450, marker='^', edgecolors='black', zorder=2)
            ax.annotate(
                "",
                xy=(phi_radians_unconstrained[i], Q_unconstrained[i]),
                xytext=(phi_radians[i], Q[i]),
                arrowprops=dict(
                    arrowstyle='->',
                    color='black',
                    lw=1
                ),
                zorder=2
            )

    ax.set_thetagrids(np.arange(0, 360, 45))
    ax.set_theta_direction(-1)
    ax.set_rmax(30)
    ax.set_rmin(7)
    ax.set_rorigin(7)
    ax.set_rticks([10, 15, 20])
    ax.set_rlabel_position(180)
    ax.tick_params(axis='y', pad=5, labelsize=16)
    ax.tick_params(axis='x', pad=18, labelsize=16)
    ax.grid(True)
    ax.text(0, ax.get_rmax() * 1.14, 'Φ',
            horizontalalignment='center', verticalalignment='center', rotation=0, fontsize=16)
    ax.text(np.pi, ax.get_rmax() * 0.975, 'Q',
            horizontalalignment='center', verticalalignment='bottom', rotation=0, fontsize=16)
            
    ticks = []
    for i in range(0, vmax+2, 2):
        ticks.append(i)

    cbar = fig.colorbar(scatter, ax=ax, ticks=ticks, orientation='horizontal', pad=0.1, fraction=0.0425)
    cbar.ax.tick_params(labelsize=30)
    symbol = get_symbol(energy_type)
    cbar.set_label(f'\u0394{symbol} (kcal/mol)', fontsize=30, labelpad=10)
    scatter.set_clim(vmin, vmax)
    ax.set_title(title, fontsize=34, pad=20)
    fig.tight_layout()
        
    return sorted_energy_index, pucker_states, energy

def make_movie(data, energy_type, title, graph_output_path):
    energy, phi_coords, theta_coords, pucker_states = return_numpy_6_mem(data, energy_type)

    sorted_intensity_index = sort_energy(energy)

    for i in range(len(energy)):
        fig, ax, norm_energy = generate_base_plot(phi_coords, theta_coords, energy)
        ax.set_title(title, fontsize=30, pad=20)
        
        if i == sorted_intensity_index[37] or i == sorted_intensity_index[0]:
            ax.scatter(phi_coords[i], theta_coords[i], c=energy[i], cmap='RdYlBu_r', norm=norm_energy,
                    s=350, marker='s', edgecolors='black', transform=ccrs.PlateCarree(), zorder=3)
        elif i == sorted_intensity_index[36] or i == sorted_intensity_index[1]:
            ax.scatter(phi_coords[i], theta_coords[i], c=energy[i], cmap='RdYlBu_r', norm=norm_energy,
                    s=350, marker='X', edgecolors='black', transform=ccrs.PlateCarree(), zorder=3)
        elif i == sorted_intensity_index[35] or i == sorted_intensity_index[2]:
            ax.scatter(phi_coords[i], theta_coords[i], c=energy[i], cmap='RdYlBu_r', norm=norm_energy,
                    s=400, marker='*', edgecolors='black', transform=ccrs.PlateCarree(), zorder=3)
        else:
            ax.scatter(phi_coords[i], theta_coords[i], c=energy[i], cmap='RdYlBu_r', norm=norm_energy,
                    s=350, marker='o', edgecolors='black', transform=ccrs.PlateCarree(), zorder=3)
        
        pucker = pucker_states[i]
        num = list(pucker_info_6_mem_dict.keys()).index(pucker)
        label = pucker_info_6_mem.loc[pucker, 'Pucker State']
        if pucker == '1C4' or pucker == '3E':
            plt.annotate(label, (phi_coords[i], theta_coords[i]), textcoords="offset points",
                        xytext=(0, -22), ha='center', va='center', fontsize=18, transform=ccrs.PlateCarree())
        elif pucker == '3H2':
            plt.annotate(label, (phi_coords[i], theta_coords[i]), textcoords="offset points",
                        xytext=(12, -23), ha='center', va='center', fontsize=18, transform=ccrs.PlateCarree())
        elif pucker == 'E2':
            plt.annotate(label, (phi_coords[i], theta_coords[i]), textcoords="offset points",
                        xytext=(-5, 21), ha='center', va='center', fontsize=18, transform=ccrs.PlateCarree())
        else:
            plt.annotate(label, (phi_coords[i], theta_coords[i]), textcoords="offset points",
                        xytext=(0, 21), ha='center', va='center', fontsize=18, transform=ccrs.PlateCarree())
            
        save_fig('png', graph_output_path, f'{num}_xylose_{pucker}')

In [ ]:
def histogram_6_mem_all_energies(path):
    colors = ['#313695', '#4575b4', '#74add1', '#a50026', '#d73027', '#f46d43']
    ax = pucker_info_6_mem.plot(kind='bar', x='Pucker State', y=['Lowest Energy #1', 'Lowest Energy #2', 'Lowest Energy #3',
                                                    'Highest Energy #1', 'Highest Energy #2', 'Highest Energy #3'],
                                                    stacked=True, width=0.8, rot=90, figsize=(34,14), color=colors, fontsize=40)
    ax.legend().remove()
    ax.set_yticks([0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20])
    ax.tick_params(axis='y', labelsize=40)
    ax.set_ylim(0, 21)
    ax.set_ylabel('Count', fontsize=40, labelpad=15)
    ax.set_xlabel('Pucker State', fontsize=40, labelpad=15)
    ax.set_xlim(-2, len(pucker_info_6_mem) + 1)

    for axis in ['top', 'bottom', 'left', 'right']:
        ax.spines[axis].set_linewidth(2)

    ax.tick_params(axis='both', which='major', width=2, length=10)
    plt.tight_layout()

    png_path = os.path.join(path, 'all_6_mem_energies.png')
    plt.savefig(png_path, dpi=300, transparent=True)
    svg_path = os.path.join(path, 'all_6_mem_energies.svg')
    plt.savefig(svg_path, format='svg', transparent=True)

def histogram_5_mem_all_energies(path):
    colors = ['#313695', '#4575b4', '#74add1', '#a50026', '#d73027', '#f46d43']
    pucker_info_5_mem.plot.bar(x='Pucker State', y=['Lowest Energy #1', 'Lowest Energy #2', 'Lowest Energy #3',
                                                    'Highest Energy #1', 'Highest Energy #2', 'Highest Energy #3'],
                                                    stacked=True, rot=70, figsize=(20,10), color=colors, fontsize=18)
    plt.title('Total Most and Least Stable Pucker States for All 5-Membered Rings', fontsize=30)
    plt.yticks([0, 1, 2, 3, 4], fontsize=18)
    plt.ylim(0, 4)
    plt.ylabel('Count', fontsize=18)
    plt.xlabel('Pucker State', fontsize=18)
    plt.legend(fontsize=18)
    
    png_path = os.path.join(path, 'all_5_mem_energies.png')
    plt.savefig(png_path, dpi=300, transparent=True)
    svg_path = os.path.join(path, 'all_5_mem_energies.svg')
    plt.savefig(svg_path, format='svg', transparent=True)

def graph_puckers(sugars, title, plot_num, puckers_formatted):
    num_boxplots = len(sugars.keys())
    if num_boxplots == 2:
        minx = 2
        maxx = 3
    elif num_boxplots == 3:
        minx = 1.5
        maxx = 3.5
    else:
        minx = 0
        maxx = 5
    positions = np.linspace(minx, maxx, num_boxplots)
    
    plot_data = pd.DataFrame.from_dict(sugars, orient='index').T.melt(var_name='Name', value_name='Energy')

    names = plot_data['Name'].unique()
    sugar_names = [titles[name] for name in names]

    fig, ax = plt.subplots(figsize=(15, 10))

    data_by_group = [
        plot_data.loc[plot_data['Name'] == name, 'Energy']
        for name in names
    ]

    vp = ax.violinplot(
        data_by_group,
        positions=positions,
        widths=0.8,
        showmeans=False,
        showextrema=False,
        showmedians=False
    )

    for i, body in enumerate(vp['bodies']):
        body.set_facecolor('#e1dfdf')   # fill color
        body.set_edgecolor('black')   # outline color
        body.set_linewidth(1.5)
        body.set_alpha(0.6)

    boxes = ax.boxplot(
        data_by_group,
        positions=positions,
        widths=0.4,
        whis=(0, 100),
        showfliers=False,
        patch_artist=True,
        boxprops=dict(
            edgecolor='black',
            linewidth=2
        ),
        whiskerprops=dict(color='black', linewidth=2),
        capprops=dict(color='black', linewidth=2),
        medianprops=dict(color='black', linewidth=2)
    )

    colors = ['#04d3ff', '#1f38ff', '#f7a502', '#02b10e', '#ff0002']
    for box, color in zip(boxes['boxes'], colors):
        box.set_facecolor(color)
        box.set_alpha(0.8)

    for axis in ['top', 'bottom', 'left', 'right']:
        ax.spines[axis].set_linewidth(2)

    i = 0
    for sugar in data_by_group:
        min = sugar.min()
        max = sugar.max()
        minid = sugar.index[sugar == min] // 11
        maxid = sugar.index[sugar == max] // 11
        ax.scatter(positions[minid], min, c='black', s=50)
        ax.scatter(positions[maxid], max, c='black')

        ax.text(positions[minid], min - 1, puckers_formatted[i], ha='center', va='center', fontsize=20)
        ax.text(positions[maxid], max + 1, puckers_formatted[i+1], ha='center', va='center', fontsize=20)
        i += 2

    ax.set_yticks([0, 5, 10, 15, 20, 25])
    ax.tick_params(axis='both', which='major', width=2, length=14, labelsize=40)
    ax.set_ylim(0, 26)
    ax.set_ylabel('\u0394G', fontsize=40, labelpad=15)
    ax.set_xlabel('')
    ax.set_xticks(positions)
    ax.set_xlim(-1, 6)
    ax.set_xticklabels(
        sugar_names,
        rotation=20,
        ha='right',
        rotation_mode='anchor',
        fontsize=40
    )

    ax.set_title(title, fontsize=44)
    plt.tight_layout()
    plt.savefig(f'/Users/sbrooks/Desktop/Structures/fig{plot_num}.svg', format='svg', transparent=True)

def alpha_vs_beta_histogram(path):
    alpha_lowest = {key: 0 for key in pucker_info_6_mem_dict}
    beta_lowest = {key: 0 for key in pucker_info_6_mem_dict}
    
    for i, sugar in enumerate(titles):
        if i >= 4:
            data = pd.read_excel(path, sheet_name=sugar)
            ten_lowest = data['Pucker State'][:10].to_list()
            
            if '\u03B1' in titles[sugar]:
                for pucker in ten_lowest:
                    alpha_lowest[pucker] += 1
            else:
                for pucker in ten_lowest:
                    beta_lowest[pucker] += 1

    graph_puckers(alpha_lowest)
    graph_puckers(beta_lowest)

    #png_path = os.path.join(path, 'all_6_mem_energies.png')
    #plt.savefig(png_path, dpi=300, transparent=True)
    #svg_path = os.path.join(path, 'all_6_mem_energies.svg')
    #plt.savefig(svg_path, format='svg', transparent=True)

def plot_pucker_transition(input_path, output_path):
    excel_path = os.path.join(input_path, 'lowest_energy_master.xlsx')
    
    northern_puckers = ['3E,', '3H4', 'E4', '5H4', '5E', '5H6', 'E6', '1H6', '1E', '1H2', 'E2', '3H2']
    equatorial_puckers = ['36B,', '3S1', 'B14', '5S1', '25B', '2S6', 'B36', '1S3', '14B', '1S5', 'B25', '6S2']
    southern_puckers = ['6E,', '6H1', 'E1', '2H1', '2E', '2H3', 'E3', '4H3', '4E', '4H5', 'E5', '6H5']

    for sugar in list(titles.keys())[4:]:
        data = pd.read_excel(excel_path, sheet_name=sugar)

        E_1C4 = data.loc[data['Pucker State'] == '1C4', 'Free Energy (kcal/mol)'].iloc[0]
        E_4C1 = data.loc[data['Pucker State'] == '4C1', 'Free Energy (kcal/mol)'].iloc[0]
        
        northern_energies = []
        equatorial_energies = []
        southern_energies = []

        data_dict = dict(zip(data['Pucker State'], data['Free Energy (kcal/mol)']))
        for pucker in data_dict:
            if pucker in northern_puckers:
                northern_energies.append(data_dict[pucker])
            elif pucker in equatorial_puckers:
                equatorial_energies.append(data_dict[pucker])
            elif pucker in southern_puckers:
                southern_energies.append(data_dict[pucker])
        
        min_northern = min(northern_energies)
        min_equatorial = min(equatorial_energies)
        min_southern = min(southern_energies)

        rev_data_dict = {v: k for k, v in data_dict.items()}

        symbol_1C4 = pucker_info_6_mem_dict['1C4'][0]
        northern_symbol = pucker_info_6_mem_dict[rev_data_dict[min_northern]][0]
        equatorial_symbol = pucker_info_6_mem_dict[rev_data_dict[min_equatorial]][0]
        southern_symbol = pucker_info_6_mem_dict[rev_data_dict[min_southern]][0]
        symbol_4C1 = pucker_info_6_mem_dict['4C1'][0]

        sugar_name = titles[sugar]
        min_barriers = [E_1C4, min_northern, min_equatorial, min_southern, E_4C1]
        puckers = [symbol_1C4, northern_symbol, equatorial_symbol, southern_symbol, symbol_4C1]

        output_graph_path = os.path.join(output_path, f'{sugar_name}.svg')
        fig, ax = plt.subplots(figsize=(8,8))
        ax.plot(puckers, min_barriers, zorder=3, color='#7286D3')
        ax.scatter(puckers, min_barriers, s=200, zorder=3, color='#7286D3')
        ax.set_ylim(0, 20)
        ax.yaxis.get_major_locator().set_params(integer=True)
        ax.set_xlim(-1, 5)
        ax.xaxis.set_label_coords(0.5, 0.95)
        ax.set_xlabel(sugar_name, fontsize=48)
        ax.set_ylabel('\u0394G (kcal/mol)', fontsize=42)
        ax.tick_params(labelsize=42)
        ax.grid(True, zorder=0, alpha=0.6)
        plt.xticks()
        plt.yticks()
        plt.tight_layout()
        fig.savefig(output_graph_path, format='svg', transparent=True)

def L_vs_D_histogram(path):
    L_lowest = {key: 0 for key in pucker_info_6_mem_dict}
    D_lowest = {key: 0 for key in pucker_info_6_mem_dict}
    
    for i, sugar in enumerate(titles):
        if i >= 4:
            data = pd.read_excel(path, sheet_name=sugar)
            ten_lowest = data['Pucker State'][:10].to_list()
            
            if '\u03B1' in titles[sugar]:
                for pucker in ten_lowest:
                    L_lowest[pucker] += 1
            else:
                for pucker in ten_lowest:
                    D_lowest[pucker] += 1

    graph_puckers(L_lowest)
    graph_puckers(D_lowest)

def get_average_energy(path):
    excel_path = os.path.join(path, 'lowest_energy_master.xlsx')
    chair_delta = {}
    all_northern = {}
    all_equatorial = {}
    all_southern = {}
    
    northern_puckers = ['3E,', '3H4', 'E4', '5H4', '5E', '5H6', 'E6', '1H6', '1E', '1H2', 'E2', '3H2']
    equatorial_puckers = ['36B,', '3S1', 'B14', '5S1', '25B', '2S6', 'B36', '1S3', '14B', '1S5', 'B25', '6S2']
    southern_puckers = ['6E,', '6H1', 'E1', '2H1', '2E', '2H3', 'E3', '4H3', '4E', '4H5', 'E5', '6H5']

    northern_symbols = []
    equatorial_symbols = []
    southern_symbols = []

    for sugar in list(titles.keys())[4:]:
        if 'galacturonic1' in sugar:
            data = pd.read_excel(excel_path, sheet_name=sugar)
            E_1C4 = data.loc[data['Pucker State'] == '1C4', 'Free Energy (kcal/mol)'].iloc[0]
            E_4C1 = data.loc[data['Pucker State'] == '4C1', 'Free Energy (kcal/mol)'].iloc[0]
            chair_delta[sugar] = abs(E_1C4 - E_4C1)

            northern_energies = []
            equatorial_energies = []
            southern_energies = []

            data_dict = dict(zip(data['Pucker State'], data['Free Energy (kcal/mol)']))
            for pucker in data_dict:
                if pucker in northern_puckers:
                    northern_energies.append(data_dict[pucker])
                elif pucker in equatorial_puckers:
                    equatorial_energies.append(data_dict[pucker])
                elif pucker in southern_puckers:
                    southern_energies.append(data_dict[pucker])
            
            all_northern[sugar] = northern_energies
            all_equatorial[sugar] = equatorial_energies
            all_southern[sugar] = southern_energies

            rev_data_dict = {v: k for k, v in data_dict.items()}

            northern_symbols.append(rev_data_dict[min(all_northern[sugar])]),
            northern_symbols.append(rev_data_dict[max(all_northern[sugar])]),
            equatorial_symbols.append(rev_data_dict[min(all_equatorial[sugar])]),
            equatorial_symbols.append(rev_data_dict[max(all_equatorial[sugar])]),
            southern_symbols.append(rev_data_dict[min(all_southern[sugar])]),
            southern_symbols.append(rev_data_dict[max(all_southern[sugar])]),

    print(northern_symbols)
    graph_puckers(all_northern, 'Northern', 1, northern_symbols)
    graph_puckers(all_equatorial, 'Equatorial', 2, equatorial_symbols)
    graph_puckers(all_southern, 'Southern', 3, southern_symbols)

In [ ]:
def write_energies(sorted_energy_index, pucker_states, energy, high_low_output_path, sugar):
    puckers = []
    energies = []
    for index in sorted_energy_index:
        puckers.append(pucker_states[index])
        energies.append(energy[index])
    output = {
        'Pucker State' : puckers,
        'Free Energy (kcal/mol)' : energies
    }
    output_df = pd.DataFrame(output)

    excel_output_path = os.path.join(high_low_output_path, 'lowest_energy_master.xlsx')
    with pd.ExcelWriter(excel_output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        output_df.to_excel(writer, sheet_name=sugar, index=False)

def save_structures_6_mem(sorted_energy_index, pucker_states, energy,
                          structure_input, high_low_output_path, sugar):
    highest_3_energy_index = sorted_energy_index[-3:][::-1]
    lowest_3_energy_index = sorted_energy_index[0:3]

    for i, index in enumerate(highest_3_energy_index):
        pucker = pucker_states[index]
        pucker_info_6_mem.loc[pucker, f'Highest Energy #{i+1}'] += 1
        structure_input_path = os.path.join(structure_input, f'{pucker}_opt.pdb')
        high_energy_sugar_path = os.path.join(high_low_output_path, 'high_energy_structures', sugar)
        os.system(f'mkdir "{high_energy_sugar_path}"')
        structure_output_path = os.path.join(high_energy_sugar_path, f'{i+1}_{pucker}_{sugar}.pdb')
        os.system(f'cp "{structure_input_path}" "{structure_output_path}"')

    for i, index in enumerate(lowest_3_energy_index):
        pucker = pucker_states[index]
        pucker_info_6_mem.loc[pucker, f'Lowest Energy #{i+1}'] += 1
        structure_input_path = os.path.join(structure_input, f'{pucker}_opt.pdb')
        low_energy_sugar_path = os.path.join(high_low_output_path, 'low_energy_structures', sugar)
        os.system(f'mkdir "{low_energy_sugar_path}"')
        structure_output_path = os.path.join(low_energy_sugar_path, f'{i+1}_{pucker}_{sugar}.pdb')
        os.system(f'cp "{structure_input_path}" "{structure_output_path}"')

        write_energies(sorted_energy_index, pucker_states, energy, high_low_output_path, sugar)

def save_structures_5_mem(sorted_energy_index, pucker_states, energy,
                          structure_input, high_low_output_path, sugar):
    highest_3_energy_index = sorted_energy_index[-3:][::-1]
    lowest_3_energy_index = sorted_energy_index[0:3]

    for i, index in enumerate(highest_3_energy_index):
        pucker = pucker_states[index]
        pucker_info_5_mem.loc[pucker, f'Highest Energy #{i+1}'] += 1
        structure_input_path = os.path.join(structure_input, f'{pucker}_opt.pdb')
        high_energy_sugar_path = os.path.join(high_low_output_path, 'high_energy_structures', sugar)
        os.system(f'mkdir "{high_energy_sugar_path}"')
        structure_output_path = os.path.join(high_energy_sugar_path, f'{i+1}_{pucker}_{sugar}.pdb')
        os.system(f'cp "{structure_input_path}" "{structure_output_path}"')

    for i, index in enumerate(lowest_3_energy_index):
        pucker = pucker_states[index]
        pucker_info_5_mem.loc[pucker, f'Lowest Energy #{i+1}'] += 1
        structure_input_path = os.path.join(structure_input, f'{pucker}_opt.pdb')
        low_energy_sugar_path = os.path.join(high_low_output_path, 'low_energy_structures', sugar)
        os.system(f'mkdir "{low_energy_sugar_path}"')
        structure_output_path = os.path.join(low_energy_sugar_path, f'{i+1}_{pucker}_{sugar}.pdb')
        os.system(f'cp "{structure_input_path}" "{structure_output_path}"')

        #write_energies(sorted_energy_index, pucker_states, energy, high_low_output_path, sugar)

In [ ]:
def graph_sugars(constrained_input, ring_atoms, output_path, energy_type, unconstrained_input):
    excel_output_path = os.path.join(output_path, 'lowest_energy_master.xlsx')
    if not(os.path.exists(excel_output_path)):
        workbook = openpyxl.Workbook()
        workbook.save(excel_output_path)
    
    for sugar in os.listdir(constrained_input):
        print(f'SUGAR: {sugar}')
        constrained_path = os.path.join(constrained_input, sugar)
        if unconstrained_input != '':
            unconstrained_path = os.path.join(unconstrained_input, sugar)

        if os.path.isdir(constrained_path):
            constrained_data_path = os.path.join(constrained_path, 'outputs', 'finished_output_energies.xlsx')
            print(f'PATH: {constrained_data_path}')
            constrained_data = pd.read_excel(constrained_data_path)
            if unconstrained_input != '':
                unconstrained_data_path = os.path.join(unconstrained_path, 'outputs', 'finished_output_energies.xlsx')
                unconstrained_data = pd.read_excel(unconstrained_data_path)
            else:
                unconstrained_data = pd.DataFrame()

            title = titles.get(sugar)
            structure_input_path = os.path.join(constrained_path, 'outputs', 'pdb_outputs')
            
            high_energy_folder = os.path.join(output_path, 'high_energy_structures')
            os.system(f'mkdir "{high_energy_folder}"')
            low_energy_folder = os.path.join(output_path, 'low_energy_structures')
            os.system(f'mkdir "{low_energy_folder}"')
            
            if ring_atoms == 5:
                sorted_energy_index, pucker_states, energy = plot_5_mem_scatter(constrained_data, unconstrained_data, energy_type, title)
                if energy_type == 'E Plus Free Energy Norm.':
                    save_structures_5_mem(sorted_energy_index, pucker_states, energy, structure_input_path, output_path, sugar)
            else:
                # Passes constrained data (data) and unconstrained data (if applicable)
                # If only plotting constrained optimization, pass an empty array as second parameter

                sorted_energy_index, pucker_states, energy = plot_6_mem_scatter(constrained_data, unconstrained_data, energy_type, title)
                if energy_type == 'E Plus Free Energy Norm.':
                    save_structures_6_mem(sorted_energy_index, pucker_states, energy, structure_input_path, output_path, sugar)

            if unconstrained_input != '':
                output_subfolder = 'unconstrained'
            elif energy_type == 'E Plus Free Energy Norm.':
                output_subfolder = 'free_energy'
            elif energy_type == 'E Plus ZPE Norm.':
                output_subfolder = 'zpe'
            elif energy_type == 'E Plus Thermal Enthalpy Norm.':
                output_subfolder = 'enthalpy'
            else:
                output_subfolder = 'spe'

            save_fig('both', output_path, sugar, output_subfolder)

# Generate energy graph for a single sugar
def graph_single_sugar(constrained_path, unconstrained_path, output_path, sugar, ring_atoms):
    constrained_data = pd.read_excel(constrained_path)
    title = titles.get(sugar)

    if unconstrained_path != '':
        unconstrained_data = pd.read_excel(unconstrained_path)

        if ring_atoms == 5:
            sorted_energy_index, pucker_states, energy = plot_5_mem_scatter(constrained_data, unconstrained_data, 'E Plus Free Energy Norm.', title)
        else:
            sorted_energy_index, pucker_states, energy = plot_6_mem_scatter(constrained_data, unconstrained_data, 'E Plus Free Energy Norm.', title)

        save_fig('both', output_path, sugar)
    else:
        unconstrained_data = pd.DataFrame()
        if ring_atoms == 5:
            sorted_energy_index, pucker_states, energy = plot_5_mem_scatter(constrained_data, unconstrained_data, 'E Plus ZPE Norm.', title)
        else:
            sorted_energy_index, pucker_states, energy = plot_6_mem_scatter(constrained_data, unconstrained_data, 'E Plus Free Energy Norm.', title)

        save_fig('both', output_path, sugar)

In [ ]:
#graph_single_sugar('/Users/sbrooks/Documents/pucker_creator/NEW_BEST_OPTS_output/final_constrained/test.xlsx', '', '/Users/sbrooks/Desktop/temp', 'galacturonic134',  6)
graph_sugars('/Users/sbrooks/Documents/pucker_creator/NEW_BEST_OPTS_output/final_constrained/5_mem', 5, '/Users/sbrooks/Desktop/Figures/energy_plots/free_energy_new', 'E Plus Free Energy Norm.', '/Users/sbrooks/Documents/pucker_creator/NEW_BEST_OPTS_output/final_unconstrained/5_mem')

In [ ]:
histogram_6_mem_all_energies('/Users/sbrooks/Desktop/Figures/energy_plots')

In [ ]:
phi = np.array([216, 234, 198, 72, 54, 90, 288, 270, 306, 144, 126, 162, 0, 18, 342, 36, 252, 108, 324, 180])
phi_radians = np.radians(phi)
q = np.array([20] * 20)

fig, ax = plt.subplots(1, 1, figsize=(10,10), subplot_kw={'projection': 'polar'})

# Set default font to Helvetica
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams.update({'mathtext.default': 'regular'})

vmin = 0
vmax = 8
norm_energy = Normalize(vmin=vmin, vmax=vmax)

data_path = '/Users/sbrooks/Documents/pucker_creator/NEW_BEST_OPTS_output/final_constrained/5_mem/apiose/outputs/finished_output_energies.xlsx'
data = pd.read_excel(data_path)
energy, phi_coords, theta_coords, pucker_states = return_numpy_5_mem(data, 'E Plus Free Energy Norm.')
print(pucker_states)

ax.set_thetagrids(np.arange(0, 360, 18))
ax.set_theta_direction(-1)
ax.set_rmax(25)
ax.set_rmin(7)
ax.set_rorigin(7)
ax.tick_params(axis='y', pad=5, labelsize=24)
ax.tick_params(axis='x', pad=18, labelsize=24)
ax.grid(True)
ax.set_rgrids([])
ax.xaxis.grid(True, color='black')
ax.yaxis.grid(True, color='black')

ax.text(0, ax.get_rmax() * 1.18, 'Φ',
        horizontalalignment='center', verticalalignment='center', rotation=0, fontsize=24)
ax.text(np.pi, ax.get_rmax() * 0.97, 'Q',
        horizontalalignment='center', verticalalignment='bottom', rotation=0, fontsize=24)

for gridline in ax.xaxis.get_gridlines():
    gridline.set_zorder(0)

for i in range(len(phi_radians)):
        pucker = pucker_states[i]
        label = pucker_info_5_mem.loc[pucker, 'Pucker State']
        plt.annotate(
             label,
             (phi_radians[i], q[i]),
             ha='center',
             va='center',
             fontsize=32,
             c='red',
             bbox=dict(
                  facecolor='white',
                  edgecolor='none',
                  boxstyle='square,pad=0.2'
                ),
             zorder=10
        )
        
ticks = []
for i in range(0, vmax+2, 2):
    ticks.append(i)

fig.tight_layout()

#plt.savefig(f'/Users/sbrooks/Desktop/furanose.svg', format='svg', bbox_inches='tight', transparent=True)

In [ ]:
phi = [0, 360, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 0]
phi = [deg-165 for deg in phi]
theta = [180, 125, 129, 125, 129, 125, 129, 125, 129, 125, 129, 125, 129, 90, 92, 90, 92, 90, 92, 90, 88, 90, 88, 90, 88, 55, 51, 55, 51, 55, 51, 55, 51, 55, 51, 55, 51, 0]
theta = [deg-90 for deg in theta]

mollweide_proj = ccrs.Mollweide()
mollweide_proj._threshold /= 100.0
fig, ax = plt.subplots(figsize=(10,10), subplot_kw={'projection': mollweide_proj})
gl = ax.gridlines(draw_labels=False, x_inline=True, xpadding=0, ypadding=0, zorder=1, color='black', alpha=0.5)
gl.xlocator = plt.FixedLocator([-165, -135, -105, -75, -45, -15, 15, 45, 75, 105, 135, 165])
[-180, -150, -120, -90, -60, -30, 0, 30, 60, 90, 120, 150, 180]
gl.ylocator = plt.FixedLocator(np.arange(-90, 91, 30))

# Annotate longitude labels
plt.annotate('0°', (-165, -56), ha='center', va='center', fontsize=20, transform=ccrs.PlateCarree())
plt.annotate('60°', (-105, -56), ha='center', va='center', fontsize=20, transform=ccrs.PlateCarree())
plt.annotate('120°', (-45, -56), ha='center', va='center', fontsize=20, transform=ccrs.PlateCarree())
plt.annotate('180°', (15, -56), ha='center', va='center', fontsize=20, transform=ccrs.PlateCarree())
plt.annotate('240°', (75, -56), ha='center', va='center', fontsize=20, transform=ccrs.PlateCarree())
plt.annotate('300°', (135, -56), ha='center', va='center', fontsize=20, transform=ccrs.PlateCarree())
plt.annotate('\nΦ', (0, -84), ha='center', va='top', fontsize=20, transform=ccrs.PlateCarree())

# Annotate latitude labels
plt.annotate('30°    ', (-180, -60), ha='right', va='center', fontsize=20, transform=ccrs.PlateCarree())
plt.annotate('60°  ', (-180, -30), ha='right', va='center', fontsize=20, transform=ccrs.PlateCarree())
plt.annotate('90° ', (-180, 0), ha='right', va='center', fontsize=20, transform=ccrs.PlateCarree())
plt.annotate('120° ', (-180, 30), ha='right', va='center', fontsize=20, transform=ccrs.PlateCarree())
plt.annotate('150°  ', (-180, 60), ha='right', va='center', fontsize=20, transform=ccrs.PlateCarree())
plt.annotate(' θ', (180, 0), ha='left', va='center', fontsize=20, transform=ccrs.PlateCarree())

# Set default font to Helvetica
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams.update({'mathtext.default': 'regular'})

for i, pucker in enumerate(pucker_info_6_mem['Pucker State'].to_numpy()):
    plt.annotate(pucker,
             (phi[i], theta[i]),
             ha='center',
             va='center',
             fontsize=24,
             c='red',
             bbox=dict(
                  facecolor='white',
                  edgecolor='none',
                  boxstyle='square, pad=0.05'
                ),
             zorder=10,
             transform=ccrs.PlateCarree())
    
plt.tight_layout()
#plt.savefig(f'/Users/sbrooks/Desktop/pyranose.svg', format='svg', bbox_inches='tight', transparent=True)